In [24]:
import pandas as pd

results = pd.read_csv("../all_results.csv")

results.head()

,Query,Method,Similarity Score (%),Text,Psalm Num,Verse
0,Create in me a clean heart,TF-IDF + GloVe,16.00,Bible,79,1For the End concerning things that shall be c...
1,Create in me a clean heart,TF-IDF + GloVe,13.98,Bible,122,1An ode of ascents Ilift my eyes to You Whodwe...
2,Create in me a clean heart,TF-IDF + GloVe,13.46,Bible,56,1For the End corrupt not by David for a pillar...
3,Create in me a clean heart,TF-IDF + GloVe,13.11,Bible,119,An ode of ascents To the Lord in my affliction...
4,Create in me a clean heart,TF-IDF + GloVe,12.74,Bible,11,For the End concerning the eighth a psalm by D...


In [25]:
results['Rank'] = (
    results.groupby(['Query', 'Method'])['Similarity Score (%)']
    .rank(method='dense', ascending=False)
    .astype(int)
)

results = results[
    ['Query', 'Method', 'Rank', 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse']
]

results['Scored'] = False

results.head(13)

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored
0,Create in me a clean heart,TF-IDF + GloVe,1,16.00,Bible,79,1For the End concerning things that shall be c...,False
1,Create in me a clean heart,TF-IDF + GloVe,2,13.98,Bible,122,1An ode of ascents Ilift my eyes to You Whodwe...,False
2,Create in me a clean heart,TF-IDF + GloVe,3,13.46,Bible,56,1For the End corrupt not by David for a pillar...,False
3,Create in me a clean heart,TF-IDF + GloVe,4,13.11,Bible,119,An ode of ascents To the Lord in my affliction...,False
4,Create in me a clean heart,TF-IDF + GloVe,5,12.74,Bible,11,For the End concerning the eighth a psalm by D...,False
5,Create in me a clean heart,BERT,1,69.89,Bible,18,For the End a psalm by David The heavens decla...,False
6,Create in me a clean heart,BERT,2,69.06,Psalter,34,"Judge Thou, O Lord, them that do me injustice;...",False
7,Create in me a clean heart,BERT,3,68.69,Psalter,49,"The God of gods, even the Lord, hath spoken, a...",False
8,Create in me a clean heart,BERT,4,68.66,Psalter,43,"We have heard with our ears, O God, for our fa...",False
9,Create in me a clean heart,BERT,5,68.31,Bible,147,Alleluia of Aggeus and Zacharias Praise the Lo...,False


In [26]:
results.shape

(985, 8)

In [27]:
queries = results[['Query']].drop_duplicates().reset_index(drop=True)

queries.head()

,Query
0,Create in me a clean heart
1,For the Peace of the world
2,"Have mercy on me, O God, have mercy on me. For..."
3,How does the psalmist express trust in God whi...
4,mercy


## Notes for generating the Pairwise assessment

1. Grab a random query
2. collect the number of remaing results (out of 5) that are still unscored
    1. randomly pick one of the possible number ranks avialable
3. collect the numbered results from each of the four possible methods
4. generate promt for an llm to be able to score each induvial result from the given query
    1. randomzie the order of the results so that order does not affect scoring at all (i.e. ensuring the same method is not always first incase the llm has bias in the order the results)
5. collect the scores and record them. 

In [28]:
import numpy as np
import os 

def get_results():
    # Getting a random query that is still available
    available_queries = results[results['Scored'] != True]['Query'].unique()
    
    query = np.random.choice(available_queries)

    # Filtering for the query
    targets = results[results['Query'] == query]

    # Pulling a random rank
    ranks = targets['Rank'].unique()
    random_rank = np.random.choice(ranks)

    # Getting rows to score
    assess_data = results[
        (results['Query'] == query) & 
        (results['Rank'] == random_rank)
    ]

    # Mark rows as scored
    results.loc[assess_data.index, 'Scored'] = True

    return assess_data

In [29]:
# results[results['Scored'] == True]

In [57]:
def build_prompt(data):

    target_query = data['Query'].unique()[0]

    prompt = ""

    if not isinstance(target_query, str):
        print("something went wrong")
    else:
        letters = [chr(65 + i) for i in range(len(data))]

        prompt += f"Query:\n{target_query}\n"
        print(f"Query:\n{target_query}")

        for i in range(len(data)):
            prompt += "Result " + letters[i] + ". ======= \n" + data['Verse'].iloc[i] + "\n\n"
            print(f"Result {letters[i]}. ======= \n{data['Verse'].iloc[i]}\n")

    print('=' * 40)
    return data, target_query, letters, prompt

In [60]:
def get_scores(data, query, letters, prompt):

    while True:
        print(f"Rank Passages {letters}, in order of relevance to the query:\n{query}")
        ordering = input(f"Rank {letters} separated by commas: ")

        letters_cleaned = [x.strip().upper() for x in ordering.split(",")]

        if set(letters_cleaned) != set(letters) or len(letters_cleaned) != len(letters):
            print("Invalid ranking.")
        else:
            break

    # Adding to the prompt
    prompt += f"""
Rank Passages {letters}, in order of relevance to the query:
{query}

Rank {letters} separated by commas:
    """

    # Assign scores based on number of passages
    scores = list(range(len(letters), 0, -1))

    score_dict = {
        letter: score
        for letter, score in zip(letters_cleaned, scores)
    }

    return score_dict, data, prompt

In [61]:
def main():
    # Generating the results to be scored in the pairwise manner
    data = get_results()
    
    # Building the prompt
    data_df, query, letters, llm_prompt = build_prompt(data)

    # Recording the scores
    score_dict, data, full_prompt = get_scores(data_df, query, letters, llm_prompt)
    
    new_scores = pd.DataFrame(
    score_dict.items(),
    columns=["Letter", "Score"]
    )

    data["Score"] = new_scores["Score"].values

    file = "llm_results.csv"
    data.to_csv(
        file, 
        mode="a",
        index=False,
        header=not os.path.exists(file)
    )
    
    print(f"Scores uploaded to {file}")
    
    
    print(f"Full Prompt for the LLM: \n {full_prompt}")
main()

Query:
Blessed is God, who illuminate and sanctifieth every man that cometh into the world: now and ever, and unto ages of ages. 
Result A. ======= 
1For the End concerning things that shall be changed a testimony for Asaph a psalm  concerning the Assyrian Give heed O You who shepherd Israel Reveal Yourself O You who lead Joseph like a flock Who sit upon the cherubim Raise up Your power Before Ephraim Benjamin and Manasseh And come for our salvation O God convert us And reveal Your face and we shall be saved O Lord God of hosts How long will You be angry with the prayer of Your servant Will You feed us the bread of tears And will You give us as drink tears in measure You made us an offense to our neighbors And our enemies sneered at us O Lord God convert us And reveal Your face and we shall be saved Pause You removed a vineyard from Egypt You cast out the nations and planted it You prepared the way before it And You planted its roots and the earth was filled Its shade covered the mount

In [ ]:

file = "llm_results.csv"
data.to_csv(
    file, 
    mode="a",
    index=False,
    header=not os.path.exists(file)
)

- **Pairwise ranking simplifies the task** by asking the LLM to choose between only two passages at a time rather than producing a complete ranking of all results.
- **Reduces formatting errors** such as missing, duplicated, or incorrectly ordered passages that can occur in listwise ranking prompts.
- **Produces more consistent judgments** by focusing the model on a single relevance comparison at each step.
- **Allows rankings to be aggregated** from multiple binary decisions, creating a final ordering based on observed preferences.
- **Aligns with preference-learning approaches** commonly used in information retrieval and LLM-as-a-judge research.
- **Provides greater interpretability** since each ranking decision can be traced back to a specific pairwise comparison.

In [63]:
data = get_results()

data

,Query,Method,Rank,Similarity Score (%),Text,Psalm Num,Verse,Scored
713,Attend to my supplication,TF-IDF + GloVe,4,13.110000,Bible,119,An ode of ascents To the Lord in my affliction...,False
718,Attend to my supplication,BERT,4,65.740000,Bible,60,For the End in hymns by David Hear my supplica...,False
723,Attend to my supplication,SBERT,4,8.040000,Bible,151,1This is a psalm written with Davids own hand ...,False
728,Attend to my supplication,TF-IDF,4,13.579788,Psalter,60,"Hear my supplication, O God; attend unto my pr...",False


In [64]:
data['Verse']

713    An ode of ascents To the Lord in my affliction...
718    For the End in hymns by David Hear my supplica...
723    1This is a psalm written with Davids own hand ...
728    Hear my supplication, O God; attend unto my pr...
Name: Verse, dtype: str

In [78]:
from itertools import combinations

def pairwise(data):

    query = data['Query'].iloc[0]

    letters = [chr(65 + i) for i in range(len(data))]

    docs = dict(zip(letters, data['Verse']))

    scores = {letter: 0 for letter in letters}

    for l1, l2 in combinations(letters, 2):

        winner = judge_pair(
            query,
            docs[l1],
            docs[l2]
        )

        if winner is None:
            continue

        if winner == "A":
            scores[l1] += 1
        elif winner == "B":
            scores[l2] += 1

    ranking = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    ranking_letters = [r[0] for r in ranking]

    print("Scores:", scores)
    print("Ranking:", ranking_letters)

    return ranking_letters, scores

In [81]:
import re

def judge_pair(query, doc_a, doc_b, judge, retries=5):
    """
    Compare two passages and return the preferred passage label ('A' or 'B').
    """

    prompt = f"""
<s>[INST]

You are an information retrieval evaluator.

Given a query, determine which passage is more relevant.

Query:
{query}

Passage A:
{doc_a}

Passage B:
{doc_b}

Output ONLY one letter:

A = Passage A is more relevant
B = Passage B is more relevant

Do not explain your answer.

[/INST]
"""

    for attempt in range(retries):

        response = judge(
            prompt,
            max_new_tokens=5,
            do_sample=False
        )

        generated = response[0]["generated_text"].strip()

        print(f"Attempt {attempt+1}: {generated}")

        match = re.search(r"\b([AB])\b", generated)

        if match:
            return match.group(1)

    return None

In [80]:
ranking, scores = pairwise(data)

TypeError: judge_pair() missing 1 required positional argument: 'judge'